In [167]:
import pandas as pd
import joblib
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [168]:
fake_df = pd.read_csv("dataset/Fake.csv")
true_df = pd.read_csv("dataset/True.csv")

In [169]:
fake_df["label"] = 0
true_df["label"] = 1

In [170]:
fake_df = fake_df[["text", "label"]]
true_df = true_df[["text", "label"]]

In [171]:
df = pd.concat([fake_df, true_df], ignore_index=True)

print(df.shape)
print(df["label"].value_counts())

(44982, 2)
label
0    23565
1    21417
Name: count, dtype: int64


In [172]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [173]:
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dy1049\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [174]:
df["text"] = df["text"].apply(clean_text)

In [175]:
X = df["text"]
y = df["label"]

In [176]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(X)

In [177]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [178]:
pac = PassiveAggressiveClassifier(max_iter=50)

pac.fit(X_train, y_train)

,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`~sklearn.linear_model.PassiveAggressiveClassifier.partial_fit` method... versionadded:: 0.19",50
,"C C: float, default=1.0Aggressiveness parameter for the passive-aggressive algorithm, see [1].For PA-I it is the maximum step size. For PA-II it regularizes thestep size (the smaller `C` the more it regularizes).As a general rule-of-thumb, `C` should be small when the data is noisy.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, the iterations will stopwhen (loss > previous_loss - tol)... versionadded:: 0.19",0.001
,"early_stopping early_stopping: bool, default=FalseWhether to use early stopping to terminate training when validationscore is not improving. If set to True, it will automatically set asidea stratified fraction of training data as validation and terminatetraining when validation score is not improving by at least `tol` for`n_iter_no_change` consecutive epochs... versionadded:: 0.20",False
,"validation_fraction validation_fraction: float, default=0.1The proportion of training data to set aside as validation set forearly stopping. Must be between 0 and 1.Only used if early_stopping is True... versionadded:: 0.20",0.1
,"n_iter_no_change n_iter_no_change: int, default=5Number of iterations with no improvement to wait before early stopping... versionadded:: 0.20",5
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.",0
,"loss loss: str, default=""hinge""The loss function to be used:hinge: equivalent to PA-I in the reference paper.squared_hinge: equivalent to PA-II in the reference paper.",'hinge'
,"n_jobs n_jobs: int or None, default=NoneThe number of CPUs to use to do the OVA (One Versus All, formulti-class problems) computation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None


In [179]:
pred = pac.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred))

print(confusion_matrix(y_test, pred))

Accuracy : 0.9962209625430699
[[4656    7]
 [  27 4307]]


In [180]:
joblib.dump(pac, "models/model.pkl")
joblib.dump(vectorizer, "models/vectorizer.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [181]:
news = "NASA Successfully Launches New Earth Observation Satellite."

cleaned = clean_text(news)
vec = loaded_vectorizer.transform([cleaned])

print("Prediction:", loaded_model.predict(vec))

Prediction: [0]


In [182]:
news = true_df.iloc[100]["text"]   # a full real article

cleaned = clean_text(news)
vec = loaded_vectorizer.transform([cleaned])

print(loaded_model.predict(vec))

[1]


In [183]:
news = fake_df.iloc[100]["text"]   # a full fake article

cleaned = clean_text(news)
vec = loaded_vectorizer.transform([cleaned])

print(loaded_model.predict(vec))

[0]
